# WiniCari -- 06 Chatbot RAG

**Module 4 : poser des questions sur le réseau en langage naturel.**

Les opérateurs et les responsables ont souvent besoin de réponses comme *« quelle ligne a la pire couverture GPS ? »* ou *« y a-t-il eu des anomalies sur la ligne 209 le mois dernier ? »* sans écrire du SQL ni lire des tableaux de bord. Le chatbot RAG leur permet de poser des questions en langage naturel et d'obtenir des réponses fondées sur les faits.

**Architecture :**

```
Question de l'utilisateur
     |
     v
sentence-transformers (all-MiniLM-L6-v2)
     |  encoder la requête
     v
ChromaDB  <-- base de connaissances (lignes, compagnies, anomalies)
     |  top-k documents pertinents
     v
Llama 3 via Groq API
     |  réponse fondée sur le contexte récupéré
     v
Réponse en langage naturel
```

La logique se trouve dans **`src/data/rag.py`**.

In [1]:
from pathlib import Path
import sys, os
sys.path.append(str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / '.env')

import pandas as pd
from IPython.display import display, Markdown

from src.data import rag
from src.data import anomaly as an

ROOT = Path(rag.__file__).resolve().parents[2]
CHROMA_DIR = ROOT / 'data' / 'processed' / 'chroma_kb'

# Load all source data needed for the knowledge base
fa        = pd.read_parquet(ROOT / 'data' / 'processed' / 'foundation_arrivals_full.parquet')
fa['trip_start'] = pd.to_datetime(fa['trip_start'])
fa['arrival']    = pd.to_datetime(fa['arrival'])
line_dist = pd.read_parquet(ROOT / 'data' / 'processed' / 'line_distances.parquet')

# Build anomaly scores to include in the knowledge base
CFG   = an.AnomalyConfig()
trips = an.trip_features(fa, CFG)
if_model, if_mean, if_std = an.train_isolation_forest(trips, CFG)
trips_scored = an.score_trips(if_model, if_mean, if_std, trips)
anomalies = trips_scored[trips_scored['anomaly']].reset_index(drop=True)

print(f'foundation: {len(fa):,} rows | lines: {line_dist.shape[0]} | anomalies: {len(anomalies)}')

foundation: 168,481 rows | lines: 402 | anomalies: 879


## 1. Construction de la base de connaissances

On génère un document texte par ligne, par compagnie et par anomalie détectée. Ces documents sont encodés avec `all-MiniLM-L6-v2` et stockés dans ChromaDB. Toute la base de connaissances est petite (~1 300 documents), donc l'indexation ne prend que quelques secondes.

**Pourquoi ne pas indexer les lignes GPS brutes ?** Il y a 168 k lignes — trop grand pour une récupération pertinente, et le modèle serait noyé dans un contexte non pertinent. Les résumés agrégés constituent un meilleur ancrage.

In [2]:
docs, ids, metas = rag.build_knowledge_base(fa, line_dist, anomalies)
print(f'documents generated: {len(docs)}')
print(f'  line docs    : {sum(1 for m in metas if m["type"]=="line")}')
print(f'  company docs : {sum(1 for m in metas if m["type"]=="company")}')
print(f'  anomaly docs : {sum(1 for m in metas if m["type"]=="anomaly")}')
print()
print('Sample documents:')
for d in docs[:3]: print(' -', d)

documents generated: 1276
  line docs    : 394
  company docs : 3
  anomaly docs : 879

Sample documents:
 - Line 202 operated by S.R.T.K. Route length: 243.5 km. Stop coverage: full (29/29 stops geocoded). Country: Tunisia.
 - Line 215 operated by S.R.T.K. Route length: 259.4 km. Stop coverage: full (29/29 stops geocoded). Country: Tunisia.
 - Line 212 operated by S.R.T.K. Route length: 189.7 km. Stop coverage: full (22/22 stops geocoded). Country: Tunisia.


In [3]:
col, embed_model = rag.build_chroma(docs, ids, metas, CHROMA_DIR)
print(f'collection ready: {col.count()} vectors')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

ChromaDB: 1276 documents indexed at C:\Users\deadx\OneDrive\Desktop\winicari\data\processed\chroma_kb
collection ready: 1276 vectors


## 2. Vérification de la récupération

Avant de brancher le LLM, vérifier que le moteur de récupération renvoie un contexte pertinent pour quelques requêtes.

In [4]:
test_queries = [
    'Which company has the worst GPS match rate?',
    'How long is line 209?',
    'Were there any anomalies on line 217?',
]

for q in test_queries:
    print(f'Q: {q}')
    hits = rag.retrieve(q, col, embed_model, k=3)
    for i, h in enumerate(hits, 1):
        print(f'  [{i}] {h}')
    print()

Q: Which company has the worst GPS match rate?
  [1] Company S.R.T.K: operates 19 lines in the foundation dataset. 459 service days, 2560 trips reconstructed. Average GPS stop match rate: 66%.
  [2] Company S.T.S: operates 14 lines in the foundation dataset. 381 service days, 3294 trips reconstructed. Average GPS stop match rate: 75%.
  [3] Company TCV: operates 1 lines in the foundation dataset. 405 service days, 14889 trips reconstructed. Average GPS stop match rate: 83%.

Q: How long is line 209?
  [1] Line 209 operated by S.T.S. Route length: 7.1 km. Stop coverage: partial (3/5 stops geocoded). Country: Tunisia.
  [2] Line 209 operated by S.R.T.K. Route length: 192.0 km. Stop coverage: full (22/22 stops geocoded). Country: Tunisia.
  [3] Line 207 operated by S.R.T.K. Route length: 123.3 km. Stop coverage: full (16/16 stops geocoded). Country: Tunisia.

Q: Were there any anomalies on line 217?
  [1] Anomalous trip detected: line 217 (S.R.T.K), direction ALLER, date 20260530, bus 603

## 3. Pipeline RAG complet -- poser des questions

On branche maintenant la récupération à Llama 3 via Groq et on pose de vraies questions d'opérateur.

In [5]:
def ask(question: str, k: int = 5):
    result = rag.ask(question, col, embed_model, k=k)
    display(Markdown(f'**Q:** {question}'))
    display(Markdown(f'**A:** {result["answer"]}'))
    print(f'  [tokens used: {result["tokens_used"]} | context docs: {len(result["context"])}]\n')

In [6]:
ask('Which company operates the most lines in the dataset?')

**Q:** Which company operates the most lines in the dataset?

**A:** Company S.R.T.K operates the most lines in the dataset with 19 lines.

  [tokens used: 336 | context docs: 5]



In [7]:
ask('What is the GPS stop match rate for each company?')

**Q:** What is the GPS stop match rate for each company?

**A:** The GPS stop match rates for each company are:

- Company S.R.T.K: 66%
- Company S.T.S: 75%
- Company TCV: 83%

  [tokens used: 393 | context docs: 5]



In [8]:
ask('Which lines have no stop geometry and cannot be used for trip reconstruction?')

**Q:** Which lines have no stop geometry and cannot be used for trip reconstruction?

**A:** Based on the provided context, the following lines have no stop geometry and cannot be used for trip reconstruction:

- Line 201 (0/20 stops geocoded)
- Line 511 (0/2 stops geocoded)
- Line 502 (0/8 stops geocoded)
- Line 503 (0/7 stops geocoded)
- Line 504 (0/10 stops geocoded)

  [tokens used: 417 | context docs: 5]



In [9]:
ask('Were there any anomalous trips on line 217? What happened?')

**Q:** Were there any anomalous trips on line 217? What happened?

**A:** Yes, there were anomalous trips on line 217. 

The anomalous trips occurred on the following dates:

1. 20251224: Bus 6019, max stop dwell: 2055 s (34.3 min), trip duration: 370 min.
2. 20251114: Bus 6037, max stop dwell: 426 s (7.1 min), trip duration: 300 min.
3. 20260530: Bus 6030, max stop dwell: 2173 s (36.2 min), trip duration: 339 min.
4. 20260110: Bus 6030, max stop dwell: 1775 s (29.6 min), trip duration: 319 min.
5. 20260210: Bus 6042, max stop dwell: 0 s (0.0 min), trip duration: 260 min.

The anomalous trips were detected due to either excessive max stop dwell times or unusual trip durations.

  [tokens used: 634 | context docs: 5]



In [10]:
ask('What is the longest line in the network and how far does it go?')

**Q:** What is the longest line in the network and how far does it go?

**A:** The longest line in the network is Line 802 operated by EPE-TVE, with a route length of 665.2 km.

  [tokens used: 351 | context docs: 5]



## 4. Mode interactif

Posez n'importe quelle question sur le réseau ici.

In [11]:
# Uncomment and change the question to explore
# ask('How many service days did S.R.T.K operate in the dataset?')
# ask('Which lines are terminal-only and why is their route distance unreliable?')
# ask('Which anomalous trips had the longest stop dwell?')

### Conclusions & prochaines étapes

- **La récupération est rapide :** ChromaDB renvoie le contexte en < 50 ms ; l'inférence Groq ~1-2 s.
- **Réponses ancrées :** le LLM n'utilise que le contexte récupéré, pas ses données d'entraînement — il ne peut pas inventer des faits sur WiniCari qui ne sont pas dans la base de connaissances.
- **La base de connaissances est facile à étendre :** ajouter de nouveaux types de documents (résumés de retard, statistiques d'immobilisation par arrêt, revenus de billetterie) en ajoutant des générateurs dans `rag.py`.
- **Vers la production :** envelopper `rag.ask()` dans un endpoint FastAPI ; le frontend envoie une question, reçoit en retour un JSON `{answer, context}`. La collection ChromaDB est déjà persistée dans `data/processed/chroma_kb/`.

**Les 4 modules sont maintenant complets :**

| module | modèle | statut |
|---|---|---|
| 1 Retard | HistGradientBoosting (base) | terminé -- à améliorer avec LSTM + Prophet |
| 2 Repli GPS | interp. linéaire + calcul à l'estime | terminé -- à améliorer avec Kalman + LSTM |
| 3 Anomalie | Isolation Forest + Autoencodeur LSTM | terminé |
| 4 Chatbot RAG | Llama 3 (Groq) + sentence-transformers + ChromaDB | terminé |